In [5]:
import json
import os
import time
from datetime import datetime
from typing import Dict, Any, List

# Configuration for the Research File
MEMORY_FILE = "rua_research_memory.json"

In [6]:
class RUACognitiveHub:
    def __init__(self, storage_path: str = MEMORY_FILE):
        self.storage_path = storage_path
        self.long_term = self._load_from_disk()
        
        # TIER 1: Working Memory (Volatile - wipes on script restart or manual clear)
        self.working_memory = {
            "session_id": datetime.now().strftime("%Y%m%d-%H%M%S"),
            "buffer": {}, # For temporary codes, links, or task-specific variables
            "active_context": None
        }

    def _load_from_disk(self) -> Dict[str, Any]:
        """Loads TIER 2 & 3 from JSON."""
        if os.path.exists(self.storage_path):
            with open(self.storage_path, "r") as f:
                return json.load(f)
        return {
            "user_profile": {"name": "Stranger", "attributes": {}},
            "semantic_memory": {}, # General facts RUA has learned
            "episodic_memory": []  # Summaries of past conversations
        }

    def save_to_disk(self):
        """Persists Long-Term data."""
        with open(self.storage_path, "w") as f:
            json.dump(self.long_term, f, indent=4)

    # --- MEMORY OPERATIONS ---

    def learn_fact(self, key: str, value: Any, persistent: bool = True):
        """Learns something new. Persistent goes to disk, non-persistent to RAM."""
        if persistent:
            self.long_term["user_profile"]["attributes"][key] = value
            self.save_to_disk()
            print(f"✨ [LTM] Learned: {key} = {value}")
        else:
            self.working_memory["buffer"][key] = value
            print(f"🧠 [STM] Buffering: {key} = {value}")

    def forget_task(self):
        """The 'Auto-Delete' mechanism for temporary data."""
        self.working_memory["buffer"] = {}
        print("🧹 [CLEANUP] Working memory wiped.")

    def get_brain_dump(self) -> str:
        """Formats all memory for the LLM System Prompt."""
        profile = self.long_term["user_profile"]
        temp = self.working_memory["buffer"]
        
        context = f"User: {profile['name']}. Known Info: {profile['attributes']}. "
        if temp:
            context += f"Current Task Data: {temp}."
        return context

# Initialize for testing
rua_brain = RUACognitiveHub()

In [7]:
def memory_router(user_input: str):
    """
    Simulation of the logic that routes input to memory.
    In the real script, this would be an LLM call.
    """
    input_lower = user_input.lower()
    
    # Logic for Identity
    if "my name is" in input_lower:
        name = user_input.split("is")[-1].strip()
        rua_brain.long_term["user_profile"]["name"] = name
        rua_brain.save_to_disk()
        return f"Got it. You're {name}. I'll remember that."

    # Logic for Temporary Storage
    if "remember this code" in input_lower:
        code = "".join(filter(str.isdigit, user_input))
        rua_brain.learn_fact("temp_code", code, persistent=False)
        return "Holding that code in my active buffer. It'll be gone when we're done."

    # Logic for Forgetting
    if any(word in input_lower for word in ["forget", "done", "clear"]):
        rua_brain.forget_task()
        return "Memory cleared. Fresh slate."

    return "Message processed."

In [8]:
# BRAIN

In [ ]:
import speech_recognition as sr
import numpy as np
from faster_whisper import WhisperModel
from langchain_ollama import ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.chat_message_histories import ChatMessageHistory
import time
import os

# --- IMPORT YOUR MEMORY LOGIC ---
# Ensure memory_management.py is in the same folder or paste the class here
# from memory_management import RUACognitiveHub, memory_router, rua_brain

# os.environ["GOOGLE_API_KEY"] = ""


def start_rua_hybrid_master():
    # --- Load Models ---
    print("👂 Powering up RUA's Ear (Faster-Whisper)...")
    stt_model = WhisperModel("small", device="cpu", compute_type="int8", cpu_threads=4)
    
    print("🏠 Initializing Local Brain (Llama 3.2)...")
    local_llm = ChatOllama(model="llama3.2", temperature=0.3)
    
    print("☁️  Initializing Cloud Brain (Gemini 1.5)...")
    # Note: Ensure your GOOGLE_API_KEY is set in your environment variables
    cloud_llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash", 
        temperature=0.3,
        google_api_key=os.environ.get("GOOGLE_API_KEY")
    )
    
    # TIER 1: Standard conversation history for the current session
    short_term_chathistory = ChatMessageHistory() 
    
    recognizer = sr.Recognizer()
    recognizer.pause_threshold = 0.8
    recognizer.non_speaking_duration = 0.5

    print("\n⚡ RUA Hybrid v6: Brain + Cognitive Memory Hub Connected.")

    try:
        with sr.Microphone(sample_rate=16000) as source:
            recognizer.adjust_for_ambient_noise(source, duration=1.0)
            
            while True:
                print("\n🟢 Listening...")
                audio = recognizer.listen(source, timeout=None, phrase_time_limit=10)
                
                # --- 1. STT STAGE ---
                raw_data = audio.get_raw_data()
                audio_np = np.frombuffer(raw_data, dtype=np.int16).astype(np.float32) / 32768.0
                segments, info = stt_model.transcribe(audio_np, beam_size=1, word_timestamps=True)
                
                detected_lang = info.language
                print(f"👤 You ({detected_lang.upper()}): ", end="", flush=True)
                
                full_user_text = ""
                for segment in segments:
                    for word in segment.words:
                        print(word.word, end="", flush=True)
                        full_user_text += word.word
                
                if full_user_text.strip():
                    # --- 2. MEMORY ROUTING (UPDATE BRAIN) ---
                    # This updates the JSON and working buffers before the LLM generates a response
                    memory_log = memory_router(full_user_text) 
                    
                    # --- 3. HYBRID ROUTING LOGIC ---
                    complex_keywords = ['research', 'explain', 'code', 'write', 'summarize', 'plan']
                    is_complex = any(k in full_user_text.lower() for k in complex_keywords)
                    
                    if is_complex:
                        selected_llm = cloud_llm
                        brain_label = "GEMINI (Cloud)"
                    else:
                        selected_llm = local_llm
                        brain_label = "LLAMA (Local)"

                    # --- 4. DYNAMIC SYSTEM PROMPT (INJECT COGNITION) ---
                    # Fetch stored facts and active task data from your memory hub
                    cognitive_context = rua_brain.get_brain_dump() 
                    
                    if detected_lang == "en":
                        lang_instruction = "Respond ONLY in English. NO Hindi."
                    else:
                        lang_instruction = "Respond in natural Hinglish (Hindi + English mix)."
                    
                    system_prompt = f"""Your name is RUA. {lang_instruction}
                    COGNITIVE CONTEXT: {cognitive_context}
                    1. Be witty and concise (under 30 words).
                    2. Use the context above to recognize the user and their tasks.
                    3. No markdown/bold."""

                    # --- 5. LLM STAGE ---
                    print(f"\n🤖 RUA [{brain_label}]: ", end="", flush=True)
                    
                    messages = [{"role": "system", "content": system_prompt}]
                    # Feed last 6 messages from conversation history
                    for msg in short_term_chathistory.messages[-6:]:
                        role = "user" if msg.type == "human" else "assistant"
                        messages.append({"role": role, "content": msg.content})
                    messages.append({"role": "user", "content": full_user_text})
                    
                    full_response = ""
                    for chunk in selected_llm.stream(messages):
                        content = chunk.content if hasattr(chunk, 'content') else str(chunk)
                        print(content, end="", flush=True)
                        full_response += content
                    
                    # Update conversation history
                    short_term_chathistory.add_user_message(full_user_text)
                    short_term_chathistory.add_ai_message(full_response)
                    
                    print(f"\n[MEM] {memory_log}") # Shows if a fact was learned or cleared
                else:
                    print("\r⚪ Silence.")

    except KeyboardInterrupt:
        print("\n🛑 RUA shutting down.")

if __name__ == "__main__":
    start_rua_hybrid_master()

👂 Powering up RUA's Ear (Faster-Whisper)...
🏠 Initializing Local Brain (Llama 3.2)...
☁️  Initializing Cloud Brain (Gemini 1.5)...

⚡ RUA Hybrid v6: Brain + Cognitive Memory Hub Connected.

🟢 Listening...
👤 You (EN):  explain line chain
🤖 RUA [GEMINI (Cloud)]: A line chain? Sounds like a queue of linked things, each waiting for the next. Like dominoes, but with a purpose... usually.
[MEM] Message processed.

🟢 Listening...
👤 You (EN):  My name is Amelie.
🤖 RUA [LLAMA (Local)]: Nice to meet you again, Amelie! So, about that line chain... I assume you're looking for info on a specific type of chain, like a necklace or a musical one?
[MEM] Got it. You're Amelie.. I'll remember that.

🟢 Listening...
⚪ Silence.): 

🟢 Listening...
👤 You (HI): 
🛑 RUA shutting down.


In [ ]:
# memory_management.py
import json
import os
from datetime import datetime
from typing import Dict, Any, List

MEMORY_FILE = "rua_multi_user_memory.json"

class RUACognitiveHub:
    def __init__(self, storage_path: str = MEMORY_FILE):
        self.storage_path = storage_path
        self.active_user = "Guest"
        self.memory = self._load_from_disk()
        
        # TIER 1: Working Memory (Session specific)
        self.working_memory = {"buffer": {}, "last_search": None}

    def _load_from_disk(self) -> Dict[str, Any]:
        if os.path.exists(self.storage_path):
            with open(self.storage_path, "r") as f:
                return json.load(f)
        return {"users": {}}

    def _ensure_user_exists(self, user_id: str):
        if user_id not in self.memory["users"]:
            self.memory["users"][user_id] = {
                "critical_info": {},
                "semantic_memory": {},
                "episodic_memory": [],
                "created_at": datetime.now().strftime("%Y-%m-%d")
            }
            self.save_to_disk()

    def set_active_user(self, user_id: str):
        self.active_user = user_id
        self._ensure_user_exists(user_id)
        print(f"👤 [IDENTITY] Switched to profile: {user_id}")

    def save_to_disk(self):
        with open(self.storage_path, "w") as f:
            json.dump(self.memory, f, indent=4)

    def learn_fact(self, key: str, value: Any, tier: str = "semantic"):
        user_data = self.memory["users"][self.active_user]
        if tier == "critical":
            user_data["critical_info"][key] = value
        else:
            user_data["semantic_memory"][key] = value
        self.save_to_disk()

    def get_brain_dump(self) -> str:
        """Returns context strictly for the active user."""
        if self.active_user == "Guest": return "User is unknown."
        
        u = self.memory["users"][self.active_user]
        dump = f"Current User: {self.active_user}. "
        if u['critical_info']: dump += f"CRITICAL: {u['critical_info']}. "
        if u['semantic_memory']: dump += f"FACTS: {u['semantic_memory']}. "
        if self.working_memory['buffer']: dump += f"ACTIVE TASK: {self.working_memory['buffer']}. "
        return dump

    def consolidate(self, llm):
        """Distills episodic memory for the active user only."""
        user_data = self.memory["users"][self.active_user]
        episodes = user_data["episodic_memory"]
        if len(episodes) < 3: return "Archives lean. No consolidation needed."
        
        # Distillation logic (same as previous version, but scoped to user)
        prompt = f"Distill these memories for {self.active_user}: {episodes}. Return JSON facts."
        try:
            res = llm.invoke(prompt)
            facts = json.loads(res.content.replace("```json", "").replace("```", "").strip())
            user_data["semantic_memory"].update(facts)
            user_data["episodic_memory"] = user_data["episodic_memory"][-1:]
            self.save_to_disk()
            return f"Brain optimized for {self.active_user}."
        except: return "Consolidation skipped."

rua_brain = RUACognitiveHub()

def identity_router(user_input: str):
    """Detects name changes or login triggers."""
    text = user_input.lower()
    if "i am" in text or "my name is" in text:
        name = user_input.split("is")[-1].strip() if "is" in text else user_input.split("am")[-1].strip()
        rua_brain.set_active_user(name.capitalize())
        return f"Identity confirmed: {rua_brain.active_user}"
    return None